# 讓 Agent 判斷什麼時候要叫工具

這份教材示範工具呼叫的流程分工：SDK 產生工具呼叫，應用層檢查參數並真正執行工具。

In [ ]:
from pathlib import Path
import os, sys, subprocess, json

if not Path('agentic_sdk').exists():
    if not Path('Agentic-SDK').exists():
        subprocess.run(['git', 'clone', 'https://github.com/R300-AI/Agentic-SDK.git'], check=True)
    os.chdir('Agentic-SDK')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## 定義工具可以接收什麼

工具 schema 是給模型看的說明；真正執行前，應用層仍然要檢查參數。

In [ ]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'create_support_ticket',
            'description': '建立客服處理單。',
            'parameters': {
                'type': 'object',
                'properties': {
                    'title': {'type': 'string', 'description': '處理單標題'},
                    'priority': {'type': 'string', 'description': '優先度'},
                },
                'required': ['title', 'priority'],
                'additionalProperties': False,
            },
        },
    }
]
tools

## 教材用的工具呼叫結果

正式使用時可用 `ToolCallAction` 讓模型產生這份結果。這裡先用固定結果示範應用層怎麼接手，讓教學可以不依賴外部 API key。

In [ ]:
latest_tool_calls = [
    {
        'id': 'call_demo_001',
        'type': 'function',
        'function': {
            'name': 'create_support_ticket',
            'arguments': json.dumps({'title': '無法登入 AI Hub', 'priority': 'high'}, ensure_ascii=False),
        },
    }
]
latest_tool_calls

In [ ]:
def create_support_ticket(title, priority):
    if priority not in {'low', 'normal', 'high'}:
        raise ValueError('priority 必須是 low、normal 或 high')
    return {'ticket_id': 'TCK-1001', 'title': title, 'priority': priority, 'status': 'created'}

for call in latest_tool_calls:
    function = call['function']
    args = json.loads(function['arguments'])
    if function['name'] == 'create_support_ticket':
        tool_result = create_support_ticket(**args)
        print(tool_result)

## 接上正式模型時

把教材中的固定 `latest_tool_calls` 換成 `ToolCallAction` 的輸出即可。請記得：SDK 不直接執行外部工具，執行權限、參數驗證與錯誤處理都由應用層負責。